## NOTE TO USER:

Running this notebook requires the additional dependency of having the executables for running MODFLOW-2005 and MT3DMS installed. See https://flopy.readthedocs.io/en/latest/md/get_modflow.html for more information. Paths to the executables should work, but can be manually set if needs be by specifying the paths with object attributes `object.exe_name_mf` and `object.exe_name_ms` for MODFLOW and MT3DMS respectively.

In [ ]:
import flopy as fp
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt
from mibitrans.analysis.to_modflow import MibitransToModflow

## Benchmarking mibitrans to numerical model

This notebook provides a comparison between the analytical solution(s) as implemented in mibitrans and a simple numerical model.
Using the `to_modflow` module of mibitrans, input parameters are converted to input for a combined MODFLOW-2005 and MT3DMS model.
Values are slightly simplified values of those used for the Keesler model (see `example_keesler.ipynb` for more information on the example parameters).

In [ ]:
hydro = mbt.HydrologicalParameters(
    h_conductivity = 9.504, # [m/d]
    h_gradient = 0.003, # [m/m]
    porosity = 0.3, # [-]
    alpha_x = 10, # [m] (Note; high dispersivity values as using same input Keesler example)
    alpha_y = 1, # [m]
    alpha_z = 0 # [m]
)
att = mbt.AttenuationParameters(
    bulk_density=1700,             # [kg/m^3]
    partition_coefficient=0.038,   # [m^3/kg] (units cancel out with bulk density)
    fraction_organic_carbon=0.000057, # [-]
    decay_rate = 0,                # [1/day]
)
source = mbt.SourceParameters(# Note; as plume is symmetric, source is described from center outwards in y-direction.
    source_zone_boundary=np.array([2, 11, 20]), # [m]
    source_zone_concentration=np.array([13.68, 2.508, 0.057]), #[g/m3]
    depth=3, # [m]
    total_mass=np.inf, # [g]
)
model = mbt.ModelParameters(
    model_length=150, # [m]
    model_width=60, # [m]
    model_time=6*365, # [days]
    dx=1, # [m]
    dy=1, # [m]
    dt=365/5 # [days]
)

In [ ]:
mbt_model = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_model.run()

Input for conversion to MODFLOW is the same as input for the analytical models. The `MibitransToModflow()` class gives an object with flopy input as attributes that can be adapted (e.g. use `object.top` to change/check the depth of the model). Using `object.to_modflow()` creates flopy `mf` and `mt` objects, containing the input parameters. Consequently, any adaptations or additions to model can be performed on these objects. In the example below, the Btn package is overwritten to include additional output times for concentrations.

In [ ]:
mbt_mf = MibitransToModflow(hydro, att, source, model)
mbt_mf.to_modflow()
mf = mbt_mf.mf
mt = mbt_mf.mt
# Note: the line below changes the mt object in place, thus, it adapts the mt object inside the mbt_mf object.
mt3d_btn = fp.mt3d.Mt3dBtn(
    mt,
    icbund=mbt_mf.icbund,
    prsity=mbt_mf.prsity,
    sconc=mbt_mf.sconc,
    nprs=4,
    timprs=[365, 2*365,4*365,6*365]
)

Use the `run_modflow()` method to run the model, which returns the concentration distribution and output times. (this step requires the MODFLOW-2005 and MT3DMS executables). Instead, input can be written to MODFLOW input files using `mf.write_input()` and/or `mt.write_input()` and be used elsewhere.

In [ ]:
conc, times = mbt_mf.run_modflow()

Use the `centerline_modflow()` method to quickly visualize the calculated concentration distribution over the plume centerline for the last timestep.

In [ ]:
mbt_mf.centerline_modflow()
plt.show()

Comparing output for mibitrans and MODFLOW, results are nearly identical, as is expected. The only noteable difference is near the model boundaries. Model boundaries are only relevant for the analytical models for what is visualized. For the numerical model constant head boundary conditions introduce a slight deviation. Picking a model extent larger than the relevant plume area prevents the already minor differences.

In [ ]:
# Determining location plume for plotting
x_pos = 80 # x-position index for transverse plot
plot_times = [1,2,4,6] # at which years to plot
x_mbt = mbt_results.x
y_mbt = mbt_results.y
model_middle = len(y_mbt)//2
row = mf.modelgrid.nrow // 2
x_mf = mf.modelgrid.xcellcenters[row, :]
y_mf = mf.modelgrid.ycellcenters[:, 0]

# Plot colors
cmap = plt.get_cmap('tab20b')
colors = cmap.colors
ci = 4
co = 2


fig, ax = plt.subplots(figsize=(10, 4), ncols=2)
for i, t in enumerate(plot_times):
    ax[0].plot(x_mbt, mbt_model.cxyt[t*5-1, model_middle, :], color=colors[ci*i+co], lw=3, label=f"t={t}y", alpha=0.3)
    ax[1].plot(y_mbt, mbt_model.cxyt[t*5-1, :, x_pos], color=colors[ci*i+co], lw=3, alpha=0.3)
for i, t in enumerate(plot_times):
    ax[0].plot(x_mf, conc[i, 0, row, :], linestyle="--", color=colors[ci*i+co], lw=3, label=" ")
    ax[1].plot(y_mf-60, conc[i, 0, :, x_pos], linestyle="--", color=colors[ci*i+co], lw=3,)
ax[1].set_xlim((-40,40))
ax[0].legend(title="   mibitrans    MODFLOW", ncol=2)
ax[0].set_xlabel("Distance from source [m]")
ax[1].set_xlabel("Distance from plume center [m]")
ax[0].set_ylabel(r"Concentration [$g/m^3$]")
plt.show()

Thus, `mibitrans` gives the same output as a simple numerical model, and offers an easy way to generate a setup of a numerical model under the same conditions.